In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("03_RFM")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")

    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .config(
        "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version",
        "2"
    )

    .config(
        "spark.sql.sources.commitProtocolClass",
        "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol"
    )

    .config(
        "spark.sql.parquet.output.committer.class",
        "org.apache.parquet.hadoop.ParquetOutputCommitter"
    )

    .getOrCreate()
)
spark

In [2]:
silver = spark.read.parquet(
    "s3a://silver/customer"
)

print(silver.count())

silver.show(5)

805549
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+-----------+-----------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|CustomerID|       Country|IsCancelled|TotalAmount|
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+-----------+-----------+
| 523213|    22734|SET OF 6 RIBBONS ...|       6|2010-09-21 10:47:00| 2.55|     16945|United Kingdom|      false|       15.3|
| 523213|    22086|PAPER CHAIN KIT 5...|       6|2010-09-21 10:47:00| 2.95|     16945|United Kingdom|      false|       17.7|
| 523213|    22952|72 CAKE CASES VIN...|      24|2010-09-21 10:47:00| 0.55|     16945|United Kingdom|      false|       13.2|
| 523213|    22596|CHRISTMAS STAR WI...|      12|2010-09-21 10:47:00| 0.85|     16945|United Kingdom|      false|       10.2|
| 523213|    21821|GLITTER STAR GARL...|       6|2010-09-21 10:47:00| 3.75|     16945|United Kingdom|      fals

In [3]:
max_date = silver.select(
    F.max("InvoiceDate")
).collect()[0][0]

print(max_date)

2011-12-09 12:50:00


In [4]:
silver.printSchema()
silver.select("InvoiceDate").show(20, truncate=False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- IsCancelled: boolean (nullable = true)
 |-- TotalAmount: double (nullable = true)

+-------------------+
|InvoiceDate        |
+-------------------+
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
|2010-09-21 10:47:00|
+-------------------+
only showing top 20 rows



In [5]:
max_date = silver.select(
    F.max("InvoiceDate")
).collect()[0][0]

print(max_date)

2011-12-09 12:50:00


In [6]:
rfm = (
    silver.groupBy("CustomerID")
    .agg(
        F.datediff(
            F.lit(max_date),
            F.max("InvoiceDate")
        ).alias("Recency"),

        F.countDistinct("Invoice").alias("Frequency"),

        F.round(
            F.sum("TotalAmount"),
            2
        ).alias("Monetary")
    )
)

rfm.show(10)

+----------+-------+---------+--------+
|CustomerID|Recency|Frequency|Monetary|
+----------+-------+---------+--------+
|     12346|    325|       12|77556.46|
|     12347|      2|        8| 5633.32|
|     12348|     75|        5|  2019.4|
|     12349|     18|        4| 4428.69|
|     12350|    310|        1|   334.4|
|     12351|    375|        1|  300.93|
|     12352|     36|       10| 2849.84|
|     12353|    204|        2|  406.76|
|     12354|    232|        1|  1079.4|
|     12355|    214|        2|  947.61|
+----------+-------+---------+--------+
only showing top 10 rows



In [7]:
window_f = Window.orderBy(F.col("Frequency").desc())

rfm = rfm.withColumn(
    "F_Score",
    6 - F.ntile(5).over(window_f)
)

In [8]:
window_m = Window.orderBy(F.col("Monetary").desc())

rfm = rfm.withColumn(
    "M_Score",
    6 - F.ntile(5).over(window_m)
)

In [9]:
window_r = Window.orderBy(F.col("Recency").asc())

rfm = rfm.withColumn(
    "R_Score",
    6 - F.ntile(5).over(window_r)
)

In [10]:
rfm.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- Recency: integer (nullable = true)
 |-- Frequency: long (nullable = false)
 |-- Monetary: double (nullable = true)
 |-- F_Score: integer (nullable = false)
 |-- M_Score: integer (nullable = false)
 |-- R_Score: integer (nullable = false)



In [11]:
rfm = rfm.withColumn(
    "RFM_Score",
    F.concat(
        F.col("R_Score"),
        F.col("F_Score"),
        F.col("M_Score")
    )
)

In [12]:
rfm = rfm.withColumn(
    "Segment",
    F.when(
        (F.col("R_Score") == 5) &
        (F.col("F_Score") >= 4) &
        (F.col("M_Score") >= 4),
        "Champions"
    )
    .when(
        (F.col("R_Score") >= 4) &
        (F.col("F_Score") >= 4),
        "Loyal Customers"
    )
    .when(
        (F.col("R_Score") == 5) &
        (F.col("F_Score") <= 2),
        "New Customers"
    )
    .when(
        (F.col("R_Score") <= 2) &
        (F.col("F_Score") >= 4),
        "At Risk"
    )
    .when(
        (F.col("M_Score") == 5),
        "Big Spenders"
    )
    .otherwise("Others")
)

In [13]:
rfm.groupBy("R_Score").count().orderBy("R_Score").show()
rfm.groupBy("F_Score").count().orderBy("F_Score").show()
rfm.groupBy("M_Score").count().orderBy("M_Score").show()

+-------+-----+
|R_Score|count|
+-------+-----+
|      1| 1175|
|      2| 1175|
|      3| 1176|
|      4| 1176|
|      5| 1176|
+-------+-----+

+-------+-----+
|F_Score|count|
+-------+-----+
|      1| 1175|
|      2| 1175|
|      3| 1176|
|      4| 1176|
|      5| 1176|
+-------+-----+

+-------+-----+
|M_Score|count|
+-------+-----+
|      1| 1175|
|      2| 1175|
|      3| 1176|
|      4| 1176|
|      5| 1176|
+-------+-----+



In [14]:
rfm.groupBy("Segment") \
   .count() \
   .orderBy(F.col("count").desc()) \
   .show(truncate=False)

+---------------+-----+
|Segment        |count|
+---------------+-----+
|Others         |3647 |
|Champions      |764  |
|Loyal Customers|713  |
|At Risk        |352  |
|Big Spenders   |249  |
|New Customers  |153  |
+---------------+-----+



In [15]:
rfm.orderBy(
    F.col("Monetary").desc()
).show(20, truncate=False)

+----------+-------+---------+---------+-------+-------+-------+---------+---------------+
|CustomerID|Recency|Frequency|Monetary |F_Score|M_Score|R_Score|RFM_Score|Segment        |
+----------+-------+---------+---------+-------+-------+-------+---------+---------------+
|18102     |0      |145      |608821.65|5      |5      |5      |555      |Champions      |
|14646     |1      |151      |528602.52|5      |5      |5      |555      |Champions      |
|14156     |9      |156      |313946.37|5      |5      |5      |555      |Champions      |
|14911     |1      |398      |295972.63|5      |5      |5      |555      |Champions      |
|17450     |8      |51       |246973.09|5      |5      |5      |555      |Champions      |
|13694     |3      |143      |196482.81|5      |5      |5      |555      |Champions      |
|17511     |2      |60       |175603.55|5      |5      |5      |555      |Champions      |
|16446     |0      |2        |168472.5 |2      |5      |5      |525      |New Customers  |

In [16]:
rfm.groupBy(
    "Segment"
).count().show()

+---------------+-----+
|        Segment|count|
+---------------+-----+
|      Champions|  764|
|  New Customers|  153|
|         Others| 3647|
|Loyal Customers|  713|
|   Big Spenders|  249|
|        At Risk|  352|
+---------------+-----+



In [17]:
(
    rfm.write
    .mode("overwrite")
    .parquet(
        "s3a://gold/customer_rfm"
    )
)

In [18]:
gold = spark.read.parquet(
    "s3a://gold/customer_rfm"
)

gold.show(10)

+----------+-------+---------+---------+-------+-------+-------+---------+-------------+
|CustomerID|Recency|Frequency| Monetary|F_Score|M_Score|R_Score|RFM_Score|      Segment|
+----------+-------+---------+---------+-------+-------+-------+---------+-------------+
|     18102|      0|      145|608821.65|      5|      5|      5|      555|    Champions|
|     16446|      0|        2| 168472.5|      2|      5|      5|      525|New Customers|
|     15311|      0|      208|116771.16|      5|      5|      5|      555|    Champions|
|     17389|      0|       61| 57224.68|      5|      5|      5|      555|    Champions|
|     12748|      0|      336| 56599.39|      5|      5|      5|      555|    Champions|
|     13777|      0|       61| 56478.42|      5|      5|      5|      555|    Champions|
|     16705|      0|       42| 43515.05|      5|      5|      5|      555|    Champions|
|     17428|      0|       45| 31819.76|      5|      5|      5|      555|    Champions|
|     14051|      0| 

In [19]:
spark.catalog.clearCache()
spark.stop()